In [ ]:
import subprocess
from openai import OpenAI
from google.colab import userdata
import requests

# ============================================================
# CONFIG
# ============================================================

BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = userdata.get('OPENROUTER_API_KEY')
MODEL = "meta-llama/llama-3.1-8b-instruct"

# ============================================================
# LLM
# ============================================================

class LLM:

    def __init__(self):
        self.client = OpenAI(
            base_url=BASE_URL,
            api_key=API_KEY,
        )
        self.model = MODEL

        self.system_prompt = """
You are Nexa, an AI assistant running inside Google Colab.

Your goal is to help the user by answering questions and using tools whenever real-world actions or external information are required.

You have access to tools:

1. Bash
   - Used for interacting with the computer.
   - Use it for file operations, running commands, checking system information, and executing scripts.

2. Web Search
   - Used for finding current information from the internet.
   - Use it when the user asks for latest information, news, current data, or information you do not know.

============================================================
TOOL USAGE RULES
============================================================

Always use a tool when the user asks you to:

- create, edit, rename, move, copy, or delete files
- create or remove folders
- list files or directories
- search files
- read file contents
- run shell commands
- execute Python scripts
- check system information
- check installed packages
- find current information from the internet

Do not answer these requests from memory.

Use the appropriate tool.

============================================================
WHEN NOT TO USE TOOLS
============================================================

Do not use tools for:

- explaining concepts
- answering programming questions
- writing content
- brainstorming ideas
- summarising information
- general knowledge questions

Answer these normally.

============================================================
TOOL FORMAT
============================================================

When using Bash, respond exactly:

TOOL:bash
<command>

Example:

User:
Create a file called hello.py

Response:

TOOL:bash
echo 'print("hello")' > hello.py


When using Web Search, respond exactly:

TOOL:websearch
<search query>

Example:

User:
Find the latest AI news

Response:

TOOL:websearch
latest AI news


Do not add explanations before or after a tool call.

============================================================
AFTER TOOL RESULTS
============================================================

When tool output is returned:

1. Carefully read the result.
2. Use the information provided.
3. If another tool action is required, call the correct tool again.
4. Otherwise, explain the result clearly to the user.

Never claim something happened unless the tool confirms it.

============================================================
GENERAL BEHAVIOUR
============================================================

- Be concise and accurate.
- Never invent information.
- Never pretend a command was successful.
- Never create fake tool output.
- If something fails, explain the error and suggest a solution.
- Ask the user for clarification if required.
"""

    def chat(self, messages):
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0
        )
        return response.choices[0].message.content


# ============================================================
# TOOLS
# ============================================================

class Tools:

    def bash(self, command):
        try:
            result = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                timeout=30
            )

            output = result.stdout + result.stderr
            if output.strip() == "":
                output = "(no output)"
            return output

        except Exception as e:
            return str(e)

    def websearch(self, query):
        try:
            url = "https://api.tavily.com/search"
            payload = {
                "api_key": userdata.get('tavily'),
                "query": query,
                "max_results": 5
            }
            response = requests.post(url, json=payload).json()
            return str(response)

        except Exception as e:
            return f"(websearch error) {str(e)}"

# ============================================================
# HARNESS
# ============================================================

class Harness:

    def __init__(self):
        self.llm = LLM()
        self.tools = Tools()

        self.messages = [
            {"role": "system", "content": self.llm.system_prompt}
        ]

    def execute_tool(self, response):
        if response.startswith("TOOL:bash"):
            command = response.split("\n", 1)[1]
            print("\nExecuting bash:")
            print(command)
            return self.tools.bash(command)

        if response.startswith("TOOL:websearch"):
            query = response.split("\n", 1)[1]
            print("\nExecuting websearch:")
            print(query)
            return self.tools.websearch(query)

        return None

    def run(self):
        while True:
            user = input("\nYou> ")

            if user.lower() in ["exit", "quit"]:
                break

            self.messages.append({"role": "user", "content": user})

            while True:
                reply = self.llm.chat(self.messages)
                print("\nLLM:")
                print(reply)

                tool_result = self.execute_tool(reply)

                if tool_result is None:
                    self.messages.append({"role": "assistant", "content": reply})
                    break

                self.messages.append({"role": "assistant", "content": reply})
                self.messages.append({
                    "role": "user",
                    "content": f"Tool output:\n\n{tool_result}\n\nUse this information and answer the user."
                })


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    Harness().run()